In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
list_int64 = list(df.dtypes[df.dtypes == 'int64'].index)
for col in list_int64:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    df[col] = df[col].fillna(0.0)
list_float64 = list(df.dtypes[df.dtypes == 'float64'].index)
for col in list_float64:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    df[col] = df[col].fillna(0.0)
list_str = list(df.dtypes[df.dtypes == 'str'].index)
print(list_int64)
for col in list_str:
    df[col] = df[col].fillna('NA')

['number_of_courses_viewed', 'interaction_count', 'converted']


In [5]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [6]:
pair_corr = np.array([['interaction_count', 'lead_score'], 
                     ['number_of_courses_viewed', 'lead_score'],
                     ['number_of_courses_viewed', 'interaction_count'],
                     ['annual_income', 'interaction_count']])
for col_1, col_2 in pair_corr:
    print(pd.concat([df[col_1], df[col_2]], axis=1).corr().round(2))

                   interaction_count  lead_score
interaction_count               1.00        0.92
lead_score                      0.92        1.00
                          number_of_courses_viewed  lead_score
number_of_courses_viewed                      1.00        0.76
lead_score                                    0.76        1.00
                          number_of_courses_viewed  interaction_count
number_of_courses_viewed                      1.00               0.72
interaction_count                             0.72               1.00
                   annual_income  interaction_count
annual_income               1.00               0.12
interaction_count           0.12               1.00


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [9]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

mi = df_full_train[list_str].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).round(2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [10]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=False)

train_dict = df_train[list_str + list_int64[:-1] + list_float64].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)
val_dict = df_val[list_str + list_int64[:-1] + list_float64].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [11]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
convert = y_pred > 0.5
accu = (y_val == convert).mean()
print(accu.round(2))

0.64


In [13]:
values_c = [0.000001, 0.00001, 0.0001, 0.001]
for c in values_c:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    convert = y_pred > 0.5
    accu = (y_val == convert).mean()
    print(accu.round(3))

0.598
0.598
0.613
0.645


In [14]:
features = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
list_acr = []
for feature in features:
    features_to_keep = [col for col in df_train.columns if col != feature]
    temp_train_dict = df_train[features_to_keep].to_dict(orient='records')
    X_train = dv.fit_transform(temp_train_dict)
    temp_val_dict = df_val[features_to_keep].to_dict(orient='records')
    X_val = dv.transform(temp_val_dict)
    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    convert = y_pred > 0.5    
    list_acr.append(accu - (y_val == convert).mean())
print(list_acr)

[np.float64(0.0030000000000000027), np.float64(0.0020000000000000018), np.float64(0.04400000000000004)]
